# 🏭 ENM412 – MAN Türkiye A.Ş. Stok Yönetimi
**Büşra ÇİL · İrem ÇELİK · Sevde SÖZDEN** | ENM412

**Kural:** A/B sınıfı → RF · XGBoost · LightGBM · CatBoost | C/Z → Hareketli Ortalama

**Adımlar:**
1. Kütüphaneleri kur
2. Dosyaları yükle
3. Pipeline çalıştır (model eğitimi)
4. Cache dosyasını indir
5. GitHub'a yükle → Streamlit Cloud'da aç

---

## 1️⃣ Kütüphane Kurulumu

In [ ]:
!pip install optuna streamlit plotly xgboost lightgbm catboost openpyxl scikit-learn scipy simpy -q
print('✅ Kurulum tamamlandı')

## 2️⃣ Dosya Yükleme

Şu dosyaları seç:
- `MAN_ML_Dataset_v3.xlsx`
- `m1_veri.py`
- `m2_modeller.py`
- `m3_optimizasyon.py`
- `m4_pipeline.py`
- `dashboard.py`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('\nYüklenen dosyalar:')
for f in uploaded:
    print(f'  ✅ {f}')

In [ ]:
import os
gerekli = ['MAN_ML_Dataset_v3.xlsx', 'm1_veri.py', 'm2_modeller.py',
           'm3_optimizasyon.py', 'm4_pipeline.py', 'dashboard.py']
eksik = []
for f in gerekli:
    ok = os.path.exists(f)
    print(f'  {"✅" if ok else "❌ EKSİK"} {f}')
    if not ok:
        eksik.append(f)
if not eksik:
    print('\n✅ Tüm dosyalar hazır!')
else:
    print(f'\n⚠️ Eksik: {eksik}')

## 3️⃣ Pipeline – Model Eğitimi

⏱️ **Tahmini süre:** 20-60 dakika

Hızlı test için `--n_trials 10` kullan.

In [ ]:
# Tam eğitim (sunum için önerilen)
!python m4_pipeline.py --dosya MAN_ML_Dataset_v3.xlsx --n_trials 30 --cache enm412_cache.pkl

In [ ]:
# Hızlı test (5-10 dk)
# !python m4_pipeline.py --dosya MAN_ML_Dataset_v3.xlsx --n_trials 10 --cache enm412_cache.pkl

In [ ]:
import os
if os.path.exists('enm412_cache.pkl'):
    boyut = os.path.getsize('enm412_cache.pkl') / 1024 / 1024
    print(f'✅ Cache hazır: {boyut:.1f} MB')
else:
    print('❌ Cache oluşmadı. Pipeline çıktısını kontrol et.')

## 4️⃣ Cache Dosyasını İndir

⚠️ **ÖNEMLİ:** Colab oturumu kapanmadan bu hücreyi çalıştır.

Dosya büyükse (>50MB) tarayıcı uyarı verebilir → **İzin Ver** de.

In [ ]:
import os
from google.colab import files

if not os.path.exists('enm412_cache.pkl'):
    print('❌ Cache bulunamadı. Önce pipeline çalıştır.')
else:
    boyut = os.path.getsize('enm412_cache.pkl') / 1024 / 1024
    print(f'📦 Cache: {boyut:.1f} MB')
    print('⬇️ İndirme başlıyor...')
    files.download('enm412_cache.pkl')
    print('✅ Bitti! GitHub reponuza yükleyin.')

## 5️⃣ GitHub → Streamlit Cloud

1. GitHub reponuza şu dosyaları yükle:
   - `enm412_cache.pkl`
   - `m1_veri.py`, `m2_modeller.py`, `m3_optimizasyon.py`
   - `m4_pipeline.py`, `dashboard.py`, `requirements.txt`
   - `MAN_ML_Dataset_v3.xlsx`

2. **share.streamlit.io** → New app → repon → `dashboard.py`

3. Deploy et → Link al → Paylaş 🎉

---

## 6️⃣ Tekil Ürün Analizi (Opsiyonel)

In [ ]:
import pickle, sys
sys.path.insert(0, '.')

with open('enm412_cache.pkl', 'rb') as f:
    sistem = pickle.load(f)

veri    = sistem['veri']
seg_mod = sistem['seg_modelleri']
batch   = sistem['batch_df']

print(f'✅ {len(veri["parcalar"]):,} parça')
print(f'ML: {len(veri["ml_parcalar"]):,} | Geleneksel: {len(veri["gel_parcalar"]):,}')
print(f'İlk 5: {veri["parcalar"][:5]}')

In [ ]:
from m2_modeller    import parca_tahmin
from m3_optimizasyon import parca_optimize, aksiyon_uyarisi
import pandas as pd
import numpy as np

PID = 'Part-1'  # İstediğin parça ID'sini yaz

# Tahmin
t = parca_tahmin(PID, veri['ml_df'], seg_mod, n_ay=6)

# Grid Search + Optuna + SimPy
o = parca_optimize(PID, veri['opt_df'], veri['abc_df'],
                   t['tahminler'], grid_adim=12, n_trials=30, n_rep=15)

# Aksiyon
uy = aksiyon_uyarisi(o, t['tahminler'])

print(f'=== {PID} ===')
print(f'Yöntem  : {"ML" if t["kullan_ml"] else "Geleneksel"} | Şampiyon: {t["sampiyon"]}')
print(f'Segment : {t["segment"]} (ABC:{t["abc"]} XYZ:{t["xyz"]})')

# Metrik — hem ML hem geleneksel metriklerine bak
tum_met  = {**t.get('ml_metrikler', {}), **t.get('gel_metrikler', {})}
samp_met = tum_met.get(t['sampiyon'], {})
print(f'MAE={samp_met.get("MAE",0):,.0f} | RMSE={samp_met.get("RMSE",0):,.0f}',
      f'| MAPE={samp_met.get("MAPE",float("nan")):.1f}%')

print(f'\nML Önerisi  : Q*={o["optimal_Q"]:,} | r*={o["optimal_r"]:,} | SS*={o["optimal_SS"]:,}')
print(f'EOQ Klasik  : Q={o["Q_eoq"]:,} | r={o["r_eoq"]:,} | SS={o["SS_eoq"]:,}')
print(f'\nEOQ Maliyet : {o["eoq_TC"]:,.0f} TL/ay (analitik)')
print(f'ML Maliyet  : {o["sim_TC"]:,.0f} TL/ay (SimPy)')
print(f'Tasarruf    : {o["tasarruf_tl"]:,.0f} TL/ay ({o["tasarruf_oran"]:.1f}%)')
print(f'Hizmet      : %{o["sim_HZ"]*100:.1f} (SimPy)')
print(f'\n{uy["mesaj"]}')

In [ ]:
# Tüm modellerin karşılaştırması
tum_met = {**t.get('ml_metrikler', {}), **t.get('gel_metrikler', {})}
rows = []
for k, v in tum_met.items():
    mape = v.get('MAPE', float('nan'))
    rows.append({'Model': k,
                 'MAE':  round(v.get('MAE',  0), 1),
                 'RMSE': round(v.get('RMSE', 0), 1),
                 'MAPE': f'{mape:.1f}%' if not np.isnan(mape) else '—',
                 'Tür':  'ML' if k in ['RF','XGBoost','LightGBM','CatBoost'] else 'Geleneksel',
                 'Şampiyon': '⭐' if k == t['sampiyon'] else ''})
print(pd.DataFrame(rows).to_string(index=False))

---
**ENM412 – Endüstri Mühendisliğinde Tasarım II**
Büşra ÇİL · İrem ÇELİK · Sevde SÖZDEN | MAN Türkiye A.Ş. | 2024-2025